In [1]:
#################################################################################
# Main script to run UP-MAVT analysis
#
# Simone Pagliuca, 2025-2026
#
# Description:
# TBC....
#################################################################################

#################################################################################
# Import third party libraries
import numpy as np
import matplotlib
matplotlib.use('TkAgg')  # Set backend before importing pyplot
import matplotlib.pyplot as plt
import seaborn as sns
import os
import csv
#################################################################################

#################################################################################
# Import internal modules
from pile_bwt import bwt, constraints_func
from up_mavt import startup, mc_simulation
from aggregation_methods import weighted_sum
from weight_sampling import create_weight_samples
#################################################################################

#################################################################################
# USER INPUTS
# Weight elicitation files (one per elicitation / run)
file_path_weight_elicitations = ["wbt_results_1.csv"]

# Value function files (one per elicitation / run)
file_path_value_functions = ["value_functions_1.csv"]

# Criteria definitions file
file_path_criteria = "criteria.csv"

# Montecarlo Parameters
n_runs = 10000
PLOTS = True  # Toggle plots
UPDATE_EVERY = 10  # Update plots every N runs
#################################################################################


#################################################################################
# Startup: Load all data
dict_data_list, crit_index, vf_list, alternatives = startup(file_path_criteria, file_path_weight_elicitations, file_path_value_functions)
n_alternatives = len(alternatives)
print(f"Loaded {len(dict_data_list)} elicitation(s) with {n_alternatives} alternatives.")
#################################################################################


Loading criteria definitions and attaching value functions...
Starting loop over weight elicitation files...
Processing file 1/1: wbt_results_1.csv
Loaded 1 elicitation(s) with 4 alternatives.


In [2]:

#################################################################################
# PILE-BWT Method
# 
# Work in progress while we try to fix issues with constraints and solver
# For now it runs the BWT optimization and then tries to sample more values from the space
print("Running BWT for each elicitation...")
bwt_results = []

for i, dict_data in enumerate(dict_data_list):
    print(f"Running BWT for elicitation {i+1}...")  # Debugging: Print elicitation index
    bwt_result = bwt(dict_data)
    bwt_results.append(bwt_result)
    
print(bwt_result["solver_result"])



Running BWT for each elicitation...
Running BWT for elicitation 1...
Starting optimization...
     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 1.2279981273413798
           x: [ 2.192e-01  6.408e-02 ...  1.252e-02  1.228e+00]
         nit: 23
         jac: [ 0.000e+00  0.000e+00 ...  0.000e+00  1.000e+00]
        nfev: 317
        njev: 21
 multipliers: [-2.877e-08  0.000e+00 ...  0.000e+00  0.000e+00]


/home/simo/GitHub/Master-Thesis/UP-MAVT/pile_bwt.py:48: RuntimeWarning: divide by zero encountered in scalar divide
  cons.append(z - abs(w[i] / w[j] - 1.0 / v_f(value)))
/home/simo/GitHub/Master-Thesis/UP-MAVT/pile_bwt.py:56: RuntimeWarning: divide by zero encountered in scalar divide
  cons.append(z - abs(1.0 / v_f_other(value) - (w[j] / w[i])))
/home/simo/GitHub/Master-Thesis/UP-MAVT/pile_bwt.py:89: RuntimeWarning: divide by zero encountered in scalar divide
  cons.append(x[-1] - abs((x[ref_global_index] / x[other_global_index]) - (1.0 / v_f(comparison['value']))))
/home/simo/GitHub/Master-Thesis/UP-MAVT/pile_bwt.py:56: RuntimeWarning: invalid value encountered in scalar divide
  cons.append(z - abs(1.0 / v_f_other(value) - (w[j] / w[i])))
/home/simo/GitHub/Master-Thesis/UP-MAVT/pile_bwt.py:48: RuntimeWarning: invalid value encountered in scalar divide
  cons.append(z - abs(w[i] / w[j] - 1.0 / v_f(value)))
/home/simo/GitHub/Master-Thesis/UP-MAVT/pile_bwt.py:89: RuntimeWarning: inval

In [3]:

print("\033[93mTesting with BWT results...\033[0m")
x_weights = bwt_result["solver_result"]["x"][:-1]
x_temp = np.concatenate([x_weights, [0]])
constraint_value = constraints_func(x_temp, dict_data)
error_b = min(max([abs(cv) for cv in constraint_value]), 10)
for c in constraint_value:
    c_py = float(c)
    print(c_py)  # Debugging: print as native Python float
print(f"\033[91mMaximum constraint violation (error_b): {error_b}\033[0m")  # Debugging
known_max_error = error_b
print("\n")
print(x_weights)


Testing with BWT results...
-0.5792513709645934
-0.48333232189805564
-0.48333232189805564
-1.227702660931214
-0.5435762061341354
-1.2279445811595047
-0.5435762061341354
-1.227901055487794
-0.5435762062277121
-1.22597531424901
-0.5435762062277121
-0.5354368479509857
-1.2270395705661619
-1.2279981269477727
-1.2270395705661619
-1.0553035732985436
-1.2279981273415586
-1.2279981273441205
-1.2278333768697647
-1.2279981273385498
-1.2279981273441205
-0.28856141597035867
-1.0128145131946502
-1.2279981272309417
-0.5327066606937043
-1.146977066380317
-1.2279981272309417
-0.05014917218476089
Maximum constraint violation (error_b): 1.2279981273441205


[0.21921743 0.06408463 0.02311608 0.04593593 0.38845369 0.08140101
 0.03797944 0.00449119 0.00657764 0.10297893 0.01324835 0.01251567]


In [4]:


print("\033[93mTesting with GA river results...\033[0m")
GA_river_results = [0.223107136295446, 0.066938158864609, 0.0240362056402704, 0.042354197922861, 0.387345646508799, 0.0745301854344023, 0.0387849189677758,0.00493893186225466, 0.00642169382682608,0.102166457316974,0.0124955605811967,0.0168809067785853]

# plot the GA river results and x_weights for comparison
indices = np.arange(len(GA_river_results))
width = 0.35
plt.bar(indices - width/2, GA_river_results, width, label='GA River Results')
plt.bar(indices + width/2, x_weights, width, label='BWT Results')
plt.xlabel('Criteria Index')
plt.ylabel('Weights')
plt.title('Comparison of GA River Results and BWT Results')
plt.xticks(indices)
plt.legend()



Testing with GA river results...


In [5]:

x_temp = np.concatenate([GA_river_results, [0]])
constraint_value = constraints_func(x_temp, dict_data)
error_b = max([abs(cv) for cv in constraint_value])
for c in constraint_value:
    c_py = float(c)
    print(c_py)  # Debugging: print as native Python float
print(f"\033[91mMaximum constraint violation (error_b): {error_b}\033[0m")  # Debugging


-0.6669663450602399
-0.2821279545741202
-0.2821279545741202
-1.2151112423310106
-0.1453897253411629
-1.2403117261211534
-0.1453897253411629
-0.8028353310656788
-1.1471038577822794
-0.9603288456769619
-1.1471038577822794
-0.6997808421081297
-0.8237796013158523
-0.9478098743735837
-0.8237796013158523
-0.6490476622563326
-1.263858104494413
-0.9870170369731248
-1.2086808461308451
-1.2475801875256423
-0.9870170369731248
-0.36582001365380323
-1.2379000014918942
-0.424421493532007
-0.6104603592898799
-0.8666809566588531
-0.424421493532007
-0.4699872503419429
Maximum constraint violation (error_b): 1.263858104494413


In [6]:

from weight_sampling import create_weight_samples, space_sampling
num_criteria = len(GA_river_results)
used_cells = set()
grid_size = np.int64(1000)
n_dimensions = num_criteria
# print("Number of dimensions for sampling:", n_dimensions)
nx = np.array([grid_size] * n_dimensions, dtype=np.int64)

initial_point = bwt_result["solver_result"]["x"][:num_criteria]
# print("Initial point for sampling:", initial_point)
initial_cell = tuple(np.clip((np.array(initial_point) * grid_size).astype(np.int64), 0, grid_size - 1))
used_cells.add(initial_cell)
valid_points = [np.array(initial_cell, dtype=np.float64) / grid_size]

print("Starting space sampling...")
print(f"Grid size: {grid_size}, Number of dimensions: {n_dimensions}")
print(f"Initial point: {initial_point}, Initial cell: {initial_cell}")


Starting space sampling...
Grid size: 1000, Number of dimensions: 12
Initial point: [0.21921743 0.06408463 0.02311608 0.04593593 0.38845369 0.08140101
 0.03797944 0.00449119 0.00657764 0.10297893 0.01324835 0.01251567], Initial cell: (np.int64(219), np.int64(64), np.int64(23), np.int64(45), np.int64(388), np.int64(81), np.int64(37), np.int64(4), np.int64(6), np.int64(102), np.int64(13), np.int64(12))


In [ ]:
print(known_max_error)

1.2279981273441205


In [ ]:
i = 0
while i < 1000:
# while True:
    i += 1
    # print(f"Iteration {i}, Used cells: {len(used_cells)}, Valid points: {len(valid_points)}")
    x_a_center = valid_points[np.random.randint(len(valid_points))]

    # Compute error_a and y_a for x_a_center
    x_temp = np.concatenate([x_a_center, [0]])
    constraints = constraints_func(x_temp, dict_data)
    error_a = min(max([abs(cv) for cv in constraints]), 10)
    # y_a = min(constraints[:num_criteria])

    # Generate a random point x_b (already sums to 1)
    x_b = np.random.dirichlet(np.ones(num_criteria), size=1)[0]

    # Compute error_b and y_b for x_b
    x_temp = np.concatenate([x_b, [0]])
    constraint_value = constraints_func(x_temp, dict_data)
    error_b = min(max([abs(cv) for cv in constraint_value]), 10)
    # print(f"Generated x_b: {x_b}, error_b: {error_b}")
    # y_b = min(constraints[:num_criteria])

    j = 0
    max_attempts = 1000
    while j < max_attempts:
        j += 1
        x_temp = np.concatenate([x_b, [0]])
        constraint_value = constraints_func(x_temp, dict_data)
        condition_1 = all(cv <= 0 for cv in constraint_value)
        condition_2 = sum(x_b) <= 1.0 + 1/grid_size and sum(x_b) >= 1.0 - 1/grid_size
        condition_3 = error_b <= known_max_error*1.1
        # print(f"abs(error_b): {abs(error_b)}", end="\r")

        if condition_1 and condition_2: #and condition_3:
            cell_idx = (x_b * grid_size).astype(int)
            cell_idx = np.minimum(cell_idx, nx - 1)
            cell = tuple(cell_idx)
            if cell not in used_cells:
                cell_center = np.array(cell_idx, dtype=np.float64) / grid_size
                valid_points.append(cell_center)
                used_cells.add(cell)
                break
        elif error_b > 100 * known_max_error:
            used_cells.add(cell)
            # Random new point
            x_b = np.random.dirichlet(np.ones(num_criteria), size=1)[0]
        else:
            direction = x_b - x_a_center
            norm = np.linalg.norm(direction)
            if norm == 0:
                x_b = np.random.dirichlet(np.ones(num_criteria), size=1)[0]
            else:
                # Line search
                alpha = 0.01
                x_b = x_a_center + alpha * direction / norm
                            
print(f"\nSpace sampling completed with {len(valid_points)} valid samples found.")


Space sampling completed with 1001 valid samples found.


In [ ]:
print(valid_points[34])
constraint_values = constraints_func(np.concatenate([valid_points[34], [0]]), dict_data)

[0.106 0.066 0.108 0.123 0.127 0.02  0.002 0.009 0.148 0.128 0.028 0.131]


In [ ]:

print("Constraint values for the 35th valid point:")
for cv in constraint_values:
    print(float(cv))

Constraint values for the 35th valid point:
-2.393939393939394
-8.018518518518519
-8.018518518518519
-3.388888888888889
-7.967479674796748
-2.83739837398374
-7.967479674796748
-0.34999999999999964
-8.777777777777779
-6.986486486486487
-8.777777777777779
-14.444444444444446
-4.428571428571429
-6.022900763358779
-4.428571428571429
-2.678571428571429
-1.8018867924528301
-54.5
-4.0078125
-46.0
-54.5
-61.0
-1.8611111111111112
-4.666666666666668
-0.39285714285714235
-8.0
-4.666666666666668
-0.1111111111111116


In [ ]:

# print("\033[93mTesting with uniform weights...\033[0m")
# constraint_value = constraints_func(np.ones(len(x_temp)), dict_data)
# for c in constraint_value:
#     c_py = float(c)
#     print(c_py)  # Debugging: print as native Python float
# #################################
# from pile_bwt import bwt_2
# bwt_result_2 = bwt_2(dict_data)
# print(bwt_result_2["solver_result"])
# input("Press Enter to continue...")

# #################################
# from pile_bwt import bwt_3
# bwt_result_3 = bwt_3(dict_data)
# print(bwt_result_3["solver_result"])
# input("Press Enter to continue...")

# #################################
# from pile_bwt import bwt_4
# result = bwt_4(dict_data, n=100, iters=3, polish=False)
# print(result["solver_result"])
# for w in result["solver_result"]["x"][:-1]:
#     print(float(w))
# constraint_value = constraints_func(result["solver_result"]["x"], dict_data)
# error_b = min(max([abs(cv) for cv in constraint_value]), 10)
# print(f"\033[91mMaximum constraint violation (error_b) for bwt_4: {error_b}\033[0m")  # Debugging
# for c in constraint_value:
#     c_py = float(c)
#     print(c_py)  # Debugging: print as native Python float
    
# input("Press Enter to continue...")









# Create files of valid sets of weights
# We have a list of errors, one per each eliciation
# We have to create tables of possible weights to sample from in the MC simulation
# weight_list = create_weight_samples(bwt_results, dict_data_list, file_path_weight_elicitations, crit_index)
# print(np.shape(weight_list))